## 🎯 Learning Objectives
* Understand the importance of handling out-of-scope queries in agentic systems.
* Learn strategies for identifying and gracefully responding to irrelevant user requests.
* Implement query classification and routing mechanisms using AutoGen agents.
* Configure AutoGen agents to terminate conversations or redirect based on query scope.


## Handling Out-of-Scope Queries Gracefully in Agentic RAG

In the world of AI assistants, especially those built for specialized tasks like e-commerce product search, users will inevitably ask questions that fall outside the system's intended capabilities. These are known as **out-of-scope queries**. Imagine walking into a high-end electronics store and asking the salesperson for a recipe for lasagna. While they might be polite, their primary function is to sell electronics, not to provide culinary advice. Similarly, an agentic RAG system designed for e-commerce product discovery should not attempt to answer questions about the weather, general history, or complex philosophical dilemmas.

### Why is Graceful Handling Crucial?

1.  **Enhanced User Experience**: A system that acknowledges its limitations and responds politely, rather than attempting to hallucinate an answer or getting stuck in a loop, provides a much better user experience. Users appreciate clarity and honesty.
2.  **Resource Optimization**: Every interaction with an LLM consumes computational resources and incurs costs. By quickly identifying and terminating out-of-scope queries, we prevent unnecessary LLM calls and agentic workflows, saving time and money.
3.  **Preventing Hallucinations and Irrelevant Responses**: When an agent tries to answer a question it's not equipped for, it's highly prone to generating incorrect, misleading, or nonsensical information (hallucinations). This erodes user trust and can lead to frustration.
4.  **Maintaining System Integrity**: Clearly defining the boundaries of an AI system helps maintain its focus and prevents it from being misused or exploited for unintended purposes.

### Strategies for Graceful Handling (2026 Perspective)

Modern agentic frameworks like AutoGen, combined with advanced LLMs, offer powerful ways to implement robust out-of-scope handling:

1.  **Intent Classification/Query Routing**: This is the primary mechanism. An initial agent (a "router" or "classifier" agent) analyzes the incoming query to determine its intent and scope. This can be done using:
    *   **LLM-based Classification**: Leveraging the reasoning capabilities of a large language model to categorize the query (e.g., "Is this query related to product search?"). This is highly flexible and adaptable.
    *   **Fine-tuned Models**: For high-throughput or very specific domains, a smaller, fine-tuned classification model (e.g., a BERT-based model) can offer faster and more cost-effective classification.
    *   **Keyword Matching/Rule-based Systems**: For simpler cases, a set of predefined keywords or rules can flag out-of-scope queries, though this is less flexible.

2.  **Guardrails and Safety Layers**: Beyond just scope, guardrails ensure queries don't violate ethical guidelines or system policies. While related, scope handling is more about functional boundaries.

3.  **Fallback Mechanisms**: If a query is deemed out-of-scope, the system should have a predefined fallback. This could be:
    *   A polite, canned response explaining the system's limitations.
    *   Redirection to a human agent or a different, more general-purpose AI.
    *   Suggesting alternative ways to phrase the query if it's borderline.

4.  **Confidence Scoring**: Advanced agents might report a confidence score for their ability to answer a query. If confidence is low, even for an in-scope query, it could trigger a fallback or human review.

### Implementing with AutoGen

AutoGen's flexible agent architecture allows us to create a dedicated `QueryClassifierAgent`. This agent acts as a gatekeeper. It receives the initial user query, uses an LLM to classify its intent, and then either:

*   **Terminates the conversation** with a polite message if the query is out-of-scope.
*   **Forwards the query** (or a refined version of it) to the appropriate specialized agent (e.g., `ProductSearchAgent`) if it's in-scope.

This modular approach ensures that specialized agents only receive relevant queries, improving efficiency and reducing the likelihood of errors.


In [ ]:
import autogen
import os

# --- Configuration --- 
# Ensure you have your OpenAI API key set as an environment variable
# For 2026, assume similar cloud-based LLM services or local high-performance models.
# Example: os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# Fallback for demonstration if API key is not set
if "OPENAI_API_KEY" not in os.environ:
    print("WARNING: OPENAI_API_KEY environment variable not set. Using a placeholder.")
    print("         This code will not run without a valid API key.")
    # Placeholder for demonstration purposes - replace with your actual key
    # In a real scenario, you would raise an error or ensure the key is present.
    os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

llm_config = {
    "config_list": [
        {
            "model": "gpt-4o", # Using a modern, capable model for classification and search
            "api_key": os.environ.get("OPENAI_API_KEY")
        }
    ],
    "temperature": 0.1 # Low temperature for more deterministic classification
}

# --- Agents Definition --- 

# 1. User Proxy Agent: Acts on behalf of the user, initiates conversations.
#    It's also responsible for determining if a message terminates the chat.
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER", # Set to NEVER for fully automated demonstration
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config=False, # No code execution needed for this demo
    llm_config=llm_config # User_Proxy can also use LLM if needed for complex orchestration
)

# 2. Query Classifier Agent: The gatekeeper. Determines if a query is in-scope.
query_classifier_agent = autogen.AssistantAgent(
    name="Query_Classifier_Agent",
    llm_config=llm_config,
    system_message=(
        "You are a highly specialized query routing agent for an e-commerce system. "
        "Your primary task is to determine if a user's query is related to searching for products "
        "in an e-commerce store. "
        "If the query is clearly about finding a product (e.g., 'find me a laptop', 'what are the best headphones', 'show me red shoes'), "
        "you MUST respond with 'IN_SCOPE: [original user query]'. "
        "If the query is NOT related to product search (e.g., 'what's the weather?', 'tell me a joke', 'how to bake a cake', 'who is the president?'), "
        "you MUST respond with 'OUT_OF_SCOPE: I can only assist with product-related inquiries. Please ask about products we sell. TERMINATE'. "
        "Your response must strictly follow one of these two formats. Do not add any other text."
    )
)

# 3. Product Search Agent: Handles in-scope product search queries.
product_search_agent = autogen.AssistantAgent(
    name="Product_Search_Agent",
    llm_config=llm_config,
    system_message=(
        "You are an expert e-commerce product search assistant. "
        "Your goal is to find products based on the user's query. "
        "Simulate a search and provide a plausible product recommendation or a summary of search results. "
        "Always end your response with 'TERMINATE'."
    )
)

# --- Group Chat Setup --- 

# The group chat will allow agents to take turns based on their roles and messages.
# The Query_Classifier_Agent will speak first to classify the query.
# If it's out-of-scope, its message will terminate the chat via user_proxy's is_termination_msg.
# If it's in-scope, the Product_Search_Agent will be prompted to respond.

groupchat = autogen.GroupChat(
    agents=[user_proxy, query_classifier_agent, product_search_agent],
    messages=[],
    max_round=12,
    speaker_selection_method="auto" # Auto selection based on turn-taking and message content
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# --- Test Cases --- 

print("\n--- Test Case 1: In-scope query ---")
user_proxy.initiate_chat(
    manager,
    message="I'm looking for a new gaming laptop with a powerful GPU."
)

print("\n--- Test Case 2: Out-of-scope query (general knowledge) ---")
user_proxy.initiate_chat(
    manager,
    message="What is the capital of France?"
)

print("\n--- Test Case 3: In-scope query (specific product) ---")
user_proxy.initiate_chat(
    manager,
    message="Show me the latest noise-cancelling headphones from Sony."
)

print("\n--- Test Case 4: Out-of-scope query (personal request) ---")
user_proxy.initiate_chat(
    manager,
    message="Can you tell me a good recipe for chocolate chip cookies?"
)

print("\n--- Test Case 5: In-scope query (broad category) ---")
user_proxy.initiate_chat(
    manager,
    message="Do you have any smart home devices on sale?"
)


### Interpreting the Code Output and Performance Considerations

When you run the provided code, you'll observe the following behaviors:

*   **For In-Scope Queries (Test Cases 1, 3, 5)**:
    1.  The `User_Proxy` sends the query to the `GroupChatManager`.
    2.  The `Query_Classifier_Agent` receives the query first. Based on its `system_message`, it uses its LLM capabilities to determine that the query is product-related.
    3.  It responds with `IN_SCOPE: [original user query]`. This message acts as a signal for the `GroupChatManager` to allow the `Product_Search_Agent` to take over.
    4.  The `Product_Search_Agent` then processes the query, simulates a product search, and provides a relevant (simulated) recommendation, ending with `TERMINATE`.
    5.  The `User_Proxy` detects the `TERMINATE` message and ends the conversation.

*   **For Out-of-Scope Queries (Test Cases 2, 4)**:
    1.  The `User_Proxy` sends the query.
    2.  The `Query_Classifier_Agent` receives the query. Its LLM determines that the query is not related to product search.
    3.  It responds with `OUT_OF_SCOPE: I can only assist with product-related inquiries. Please ask about products we sell. TERMINATE`.
    4.  The `User_Proxy` immediately detects the `TERMINATE` keyword in the `Query_Classifier_Agent`'s response and gracefully ends the conversation, preventing any further processing by the `Product_Search_Agent`.

This demonstrates a clean and effective way to route queries and terminate conversations when necessary, ensuring that only relevant queries reach the specialized agents.

### Performance Trade-offs and Use Cases

**Performance Trade-offs:**

1.  **Latency**: Introducing a `Query_Classifier_Agent` adds an additional LLM call (or a classification model inference) at the beginning of every interaction. For highly latency-sensitive applications, this overhead needs to be considered. However, for most conversational AI systems, the slight delay is acceptable given the benefits.
2.  **Cost**: Each LLM call incurs a cost. While classification prompts are usually short, an extra call per interaction can add up, especially at high volumes. Optimizations might include using a smaller, cheaper LLM for classification or a dedicated, fine-tuned classification model if the domain is very stable.
3.  **Accuracy of Classification**: The effectiveness of this approach heavily relies on the `Query_Classifier_Agent`'s ability to accurately distinguish between in-scope and out-of-scope queries. Misclassifications (false positives or false negatives) can lead to frustrated users or wasted resources. The `system_message` and the choice of LLM are critical here.

**Typical Use Cases:**

*   **Specialized Customer Support Bots**: Ensuring that a bot designed for technical support doesn't try to answer billing questions, instead redirecting them appropriately.
*   **Knowledge Base Assistants**: Guiding users to the correct section of a knowledge base or informing them if their query is outside the documented topics.
*   **E-commerce Product Discovery (as demonstrated)**: Preventing general inquiries from reaching the product search pipeline.
*   **Internal Tools**: Routing employee queries to the correct internal system or department based on intent.
*   **Safety and Content Moderation**: As a first line of defense to filter out harmful, inappropriate, or policy-violating queries before they reach core agents.


### Resources

*   **AutoGen Documentation**: Explore advanced agent capabilities, custom termination conditions, and group chat management.
    *   [AutoGen GitHub Repository](https://github.com/microsoft/autogen)
    *   [AutoGen Official Documentation](https://microsoft.github.io/autogen/docs/)
    *   [AutoGen Group Chat Tutorial](https://microsoft.github.io/autogen/docs/Use-Cases/agent_chat)

*   **LLM Safety and Guardrails**: Understand broader concepts of responsible AI development.
    *   [Google's Responsible AI Practices](https://ai.google/responsibility/)
    *   [Hugging Face's Transformers Safety Guide](https://huggingface.co/docs/transformers/main/en/llm_tutorial#safety-and-ethical-considerations)

*   **Intent Classification and NLU**: Learn more about the underlying techniques for query understanding.
    *   [NLTK (Natural Language Toolkit) for Python](https://www.nltk.org/)
    *   [SpaCy for Industrial-Strength NLP](https://spacy.io/)
    *   [Hugging Face Transformers Library](https://huggingface.co/docs/transformers/index)
